# Module 2 Exercise (Solution): Depth, gradients, and residuals on a CIFAR-10 subset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/02-cnn-refresher/exercise_solution.ipynb)

Module page: [Module 2: CNN Refresher](https://nsteve2407.github.io/llm-transformers-course/modules/02-cnn-refresher/)

**Part A**: implement `Conv2dManual`, a from-scratch 2D convolution (`unfold` + matmul), and validate it against `torch.nn.functional.conv2d`.

**Part B**: build `PlainNet(depth)` (plain conv/BN/ReLU stacks, no skip connections) and `ResNet(depth)` (same structure, with identity/projection skip connections) at depths 8, 20, and 44.

**Part C**: train all six models on a CIFAR-10 subset, recording per-step training loss and the first conv layer's gradient norm, then plot both to observe the degradation problem and gradient health.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

In [ ]:
if SMOKE_TEST:
    X_train = torch.rand(256, 3, 32, 32)
    y_train = torch.randint(0, 10, (256,))
    X_val = torch.rand(64, 3, 32, 32)
    y_val = torch.randint(0, 10, (64,))
else:
    from torchvision import datasets, transforms
    tfm = transforms.Compose([transforms.ToTensor()])
    train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=tfm)
    val_ds = datasets.CIFAR10(root="./data", train=False, download=True, transform=tfm)
    n_train = 10000
    n_val = 2000
    X_train = torch.stack([train_ds[i][0] for i in range(n_train)])
    y_train = torch.tensor([train_ds[i][1] for i in range(n_train)])
    X_val = torch.stack([val_ds[i][0] for i in range(n_val)])
    y_val = torch.tensor([val_ds[i][1] for i in range(n_val)])

print(X_train.shape, y_train.shape)

## Part A: `Conv2dManual` -- from-scratch 2D convolution, validated against `F.conv2d`

In [ ]:
class Conv2dManual(nn.Module):
    """2D convolution implemented with unfold (im2col) + matmul, no F.conv2d call."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        fan_in = in_channels * kernel_size * kernel_size
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size) * fan_in ** -0.5)
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None

    def forward(self, x):
        N, C, H, W = x.shape
        K, S, P = self.kernel_size, self.stride, self.padding
        H_out = (H + 2 * P - K) // S + 1
        W_out = (W + 2 * P - K) // S + 1

        # im2col: each column is a flattened (C*K*K) receptive-field patch.
        patches = F.unfold(x, kernel_size=K, stride=S, padding=P)  # (N, C*K*K, H_out*W_out)
        weight_flat = self.weight.view(self.out_channels, -1)  # (out_channels, C*K*K)

        # (out_channels, C*K*K) @ (N, C*K*K, L) broadcasts to (N, out_channels, L).
        out = torch.matmul(weight_flat, patches)
        if self.bias is not None:
            out = out + self.bias.view(1, -1, 1)
        return out.view(N, self.out_channels, H_out, W_out)

In [ ]:
# Validate Conv2dManual against F.conv2d on a non-trivial random input.
torch.manual_seed(0)
x_probe = torch.randn(4, 3, 17, 17)
conv_manual = Conv2dManual(in_channels=3, out_channels=8, kernel_size=3, stride=2, padding=1)

out_manual = conv_manual(x_probe)
out_ref = F.conv2d(x_probe, conv_manual.weight, conv_manual.bias, stride=2, padding=1)

max_diff = (out_manual - out_ref).abs().max().item()
print("output shape:", out_manual.shape, "max abs diff vs F.conv2d:", max_diff)
assert torch.allclose(out_manual, out_ref, atol=1e-4), "Conv2dManual does not match F.conv2d!"
print("Conv2dManual matches F.conv2d.")

## Part B: `PlainNet(depth)` vs. `ResNet(depth)` at matched depth/width

**Design choice (documented judgment call):** we use the classic CIFAR-10 ResNet parameterization from
He et al. (2015), *"Deep Residual Learning for Image Recognition"*. Total depth = `6n + 2` weight layers:
one 3x3 stem conv, three stages of `n` basic blocks each (2 convs per block) at widths 16/32/64 channels,
with stride-2 downsampling at the start of stages 2 and 3, and a final FC layer. Depths 8, 20, and 44
correspond to `n = 1, 3, 7`.

`PlainNet` and `ResNet` share the *exact same* stem/stage/channel/depth structure (`ConvNet` below) and
therefore have matched parameter counts at each depth -- the only difference is the block type: `PlainBlock`
computes `F(x)`, while `ResBlock` computes `relu(F(x) + shortcut(x))` with a 1x1-conv+BN projection shortcut
whenever channels or spatial resolution change between input and output. This isolates the skip connection
as the sole variable between the two architectures.

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, in_c, out_c, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        return out


class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


def make_stage(block_cls, in_c, out_c, n_blocks, stride):
    layers = [block_cls(in_c, out_c, stride)]
    for _ in range(n_blocks - 1):
        layers.append(block_cls(out_c, out_c, 1))
    return nn.Sequential(*layers)


class ConvNet(nn.Module):
    """Shared skeleton for PlainNet and ResNet -- only `block_cls` differs."""

    def __init__(self, depth, block_cls, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "depth must be 6n+2 (e.g. 8, 20, 44)"
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.stage1 = make_stage(block_cls, 16, 16, n, stride=1)
        self.stage2 = make_stage(block_cls, 16, 32, n, stride=2)
        self.stage3 = make_stage(block_cls, 32, 64, n, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def PlainNet(depth, num_classes=10):
    return ConvNet(depth, PlainBlock, num_classes)


def ResNet(depth, num_classes=10):
    return ConvNet(depth, ResBlock, num_classes)

## Part C: train all six models, recording loss and first-layer gradient norm

For each of Plain-8/20/44 and ResNet-8/20/44 we track, at every training step: the training loss, and the
gradient norm of the **stem conv's** weight (`model.stem[0].weight.grad.norm()`), captured right after
`.backward()` and before `optimizer.step()`. The stem is the first layer every model has in common, so its
gradient norm is a consistent proxy for "how much signal reaches the earliest layer" across all six models.

In [ ]:
model_configs = [
    ("Plain-8", PlainNet, 8),
    ("Plain-20", PlainNet, 20),
    ("Plain-44", PlainNet, 44),
    ("ResNet-8", ResNet, 8),
    ("ResNet-20", ResNet, 20),
    ("ResNet-44", ResNet, 44),
]

epochs = 1 if SMOKE_TEST else 8
batch_size = 64 if SMOKE_TEST else 128
lr = 0.01

history = {}
for name, ctor, depth in model_configs:
    torch.manual_seed(0)
    model = ctor(depth).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    losses, grad_norms = [], []
    for epoch in range(epochs):
        perm = torch.randperm(X_train.shape[0])
        for i in range(0, X_train.shape[0], batch_size):
            idx = perm[i : i + batch_size]
            xb, yb = X_train[idx].to(device), y_train[idx].to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            grad_norm = model.stem[0].weight.grad.norm().item()  # captured before opt.step()
            opt.step()
            losses.append(loss.item())
            grad_norms.append(grad_norm)
    history[name] = {"loss": losses, "grad_norm": grad_norms}
    print(f"{name}: final loss={losses[-1]:.4f}, mean stem grad norm={sum(grad_norms) / len(grad_norms):.4e}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
for name, h in history.items():
    plt.plot(h["loss"], label=name)
plt.xlabel("training step")
plt.ylabel("training loss")
plt.legend()
plt.title("Training loss: PlainNet vs. ResNet at depths 8/20/44")
plt.show()

In [ ]:
plt.figure()
for name, h in history.items():
    plt.plot(h["grad_norm"], label=name)
plt.xlabel("training step")
plt.ylabel("stem conv weight grad norm")
plt.yscale("log")
plt.legend()
plt.title("First-layer gradient norm vs. training step")
plt.show()

## Discussion

Look at the two plots above. In the original ResNet paper, plain networks get **worse** training performance
as depth increases from 8 to 44 (not just overfitting -- worse *training* loss), while ResNets at the same
depths train about as well or better as they get deeper, with healthier (non-vanishing) gradients reaching
the stem layer.

**Note whether that pattern shows up in your run.** With only a handful of epochs on a 10k-image CIFAR-10
subset, the degradation effect and the gradient-health gap may be subtle rather than dramatic -- that's
expected and fine. The point of this exercise is exposure to the phenomenon and to the diagnostic technique
(tracking early-layer gradient norms across depth), not a guaranteed dramatic separation. Try increasing
`epochs`, training set size, or depth (e.g. add a depth-110 config) if you want to chase a sharper effect.

**A note on gradients specifically:** because every block here (`PlainBlock` included) uses BatchNorm,
`PlainNet`'s stem gradient norm will likely look healthy too, even at depth 44 -- BatchNorm keeps gradients
well-scaled, so this is the *expected* outcome, not a failed reproduction. The degradation problem is an
optimization-difficulty issue (plain stacks struggle to learn even an identity mapping), not a
vanishing-gradient issue -- that's exactly the distinction quiz Q8 is testing.